# Session 5 — Pandas II, Visualization & Clean Code
### Block B · Saint Mary's General Hospital — Operations Analytics Unit

> *"By Tuesday I'm presenting to the board. I need a one-pager: occupancy, margin, ER. Make me a notebook I can read in 5 minutes."* — CMO, 08:55 Friday

This is the final session before the exam. Today we **finish** the analytics pipeline: GroupBy, Merge, Visualization. And we talk about **clean code** — what makes a notebook exam-ready.

By the end of this session you can:
1. Use **GroupBy** for split-apply-combine analytics.
2. **Merge** two tables on a key with the right join type.
3. Build a readable **Matplotlib** chart with title, labels, units.
4. Structure a notebook so the next reader (or examiner) can follow it.
5. State the **exam requirements** clearly.

---

**Setting:** Q1 closes today. The CMO wants three numbers and three pictures. By Tuesday: a notebook anyone on the board can read.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

HERE = Path.cwd()
DATA = next((p / "data" for p in [HERE, *HERE.parents] if (p / "data").exists()), None)
assert DATA is not None, "data/ folder not found"
print("Using DATA =", DATA)

# Load the three datasets we'll use today:
enc = pd.read_csv(DATA / "encounters.csv", parse_dates=["admission_date","discharge_date"])
drg = pd.read_csv(DATA / "drg_catalog.csv")
patients = pd.read_csv(DATA / "patients.csv")
print(f"encounters={len(enc)} | drg={len(drg)} | patients={len(patients)}")

## §1 — GroupBy: split-apply-combine

> Mental model: **split** the data into buckets by some key, **apply** a function to each bucket, **combine** the results into a new structure.

This is the most important pattern in tabular data analysis. Once you internalise it, you stop writing loops.

In [ ]:
# Average length of stay per ward — one line:
los_by_ward = enc.groupby("ward")["los_days"].mean().sort_values(ascending=False)
los_by_ward

> **Try it.** Compute the *maximum* length of stay per ward — which ward has the single longest case? *(Swap `.mean()` for `.max()` in the line above.)*

In [ ]:
# Multiple aggregations at once — the pandas dialect:
ward_stats = enc.groupby("ward").agg(
    n_cases=("encounter_id", "count"),
    total_cost=("actual_cost_eur", "sum"),
    mean_los=("los_days", "mean"),
    median_los=("los_days", "median"),
).sort_values("n_cases", ascending=False)
ward_stats

> **Try it.** Add `mean_cost=("actual_cost_eur", "mean")` to the `.agg(...)` call above. Which ward has the highest mean cost per case — and is it the same ward as the one with the most cases?

In [ ]:
# .transform() puts the aggregated value back on the original rows.
# Useful when you want a 'how does this case compare to its ward average' column:
enc["ward_mean_los"] = enc.groupby("ward")["los_days"].transform("mean")
enc["los_above_avg"] = enc["los_days"] > enc["ward_mean_los"]
enc[["encounter_id","ward","los_days","ward_mean_los","los_above_avg"]].head(8)

> **Try it.** Build a `cost_above_ward_avg` column with the same pattern: `transform("mean")` on `actual_cost_eur`, then compare. What share of all encounters lie above their ward's average cost? *(Hint: `.mean()` on a boolean Series returns the share of `True`.)*

## §2 — Merging tables

Real analytics joins multiple tables. Pandas's `merge()` is your friend.

**Four join types** (same as SQL):
- `how="inner"` — only rows present in both tables
- `how="left"` — all rows from left, fill with NaN if no match
- `how="right"` — all rows from right
- `how="outer"` — all rows from both, NaN where missing

In practice 90% of joins are `left`.

In [ ]:
# Encounter + DRG → margin per case (we did this in S4)
enc = enc.merge(drg, on="drg_code", how="left")
enc["margin_eur"] = enc["base_reimbursement_eur"] - enc["actual_cost_eur"]
enc[["encounter_id","drg_code","label","margin_eur"]].head()

> **Try it.** Read the `margin_eur` of the encounter in row 0 with `enc.loc[0, "margin_eur"]`. Then read the DRG `label` of the same row. *(That is `.loc` for a single value — exactly the micro-skill the project asks for.)*

In [ ]:
# Encounter + Patient → cohort analysis: are older patients more expensive?
enc = enc.merge(patients[["patient_id","age","gender","primary_insurance"]],
                on="patient_id", how="left")
enc[["encounter_id","patient_id","age","gender","actual_cost_eur"]].head()

> **Try it.** Count how many encounters belong to female versus male patients with `enc["gender"].value_counts()`. Are the counts roughly balanced?

In [ ]:
# Now we can group by an attribute that originally lived in another table:
cost_by_age_band = (enc.assign(age_band=pd.cut(enc["age"],
                                                bins=[0,40,60,75,100],
                                                labels=["<40","40-59","60-74","75+"]))
                       .groupby("age_band", observed=True)["actual_cost_eur"]
                       .agg(["mean","median","count"])
                       .round(2))
cost_by_age_band

> **Try it.** Add a fourth column `total_cost` to the result using `agg(["mean","median","count","sum"])`. Which age band drives the most absolute cost — and is it driven by case count or by mean cost per case?

## ⏸️ Mini-Challenge 1 — GroupBy (25 min)

Open `challenges/ch1_groupby.ipynb`.

**You will:** compute average LOS per ward, total revenue + average margin per DRG, sort and present. *Fast-Track:* find the ward with highest LOS variance and interpret what it tells you about case mix.

## §3 — pivot_table: when to use, when to skip

`pivot_table` is GroupBy's wide-format cousin. Use it for **display**, not for analysis.

In [ ]:
# Wide format: ward x gender matrix of mean LOS.
los_matrix = pd.pivot_table(
    enc,
    values="los_days",
    index="ward",
    columns="gender",
    aggfunc="mean",
).round(1)
los_matrix

> **Try it.** Change `aggfunc="mean"` to `aggfunc="count"`. The matrix now shows case counts. Are there ward × gender combinations with very few cases?

**Rule of thumb:** if your next step is *another aggregation*, use GroupBy. If your next step is *display in a report*, use pivot_table.

## §4 — Visualization: the OO Matplotlib API

Beginners are taught `plt.plot(...)` — the "implicit" API. **Don't.** Once your figure has more than one chart, the implicit API gets confusing.

The robust pattern is:
```
fig, ax = plt.subplots(figsize=(W, H))
ax.bar(...)
ax.set_title(...)
ax.set_xlabel(...)
plt.tight_layout()
plt.show()
```

Three lines of ceremony — and you can always extend by adding more `ax.something(...)` calls.

In [ ]:
# Bar chart: top 10 loss-making DRGs
top_loss = (enc[enc["margin_eur"] < 0]
            .groupby("drg_code")["margin_eur"]
            .sum()
            .sort_values()
            .head(10))

fig, ax = plt.subplots(figsize=(10, 6))
top_loss.plot(kind="barh", ax=ax, color="#c0392b")
ax.set_title("Top 10 loss-making DRGs · Saint Mary's Q1 2026")
ax.set_xlabel("Total margin (EUR, negative = loss)")
ax.set_ylabel("DRG code")
ax.axvline(0, color="black", linewidth=1)
plt.tight_layout()
plt.show()

> **Try it.** Modify the chart to show the top 10 *most profitable* DRGs instead. *(Drop the `< 0` filter, sort `ascending=False`, take `.head(10)`, and switch the bar colour to green `#2e8b57`.)*

In [ ]:
# Line chart: daily occupancy on Cardiology over Q1
WARD_CAPACITY = {"ICU": 16, "InternalMedicine": 60, "Surgery": 40,
                 "Cardiology": 30, "Geriatrics": 28, "Pulmonology": 20, "Neurology": 18}

dates = pd.date_range("2026-01-01", "2026-03-31", freq="D")
ward = "Cardiology"
ward_enc = enc[enc["ward"] == ward]

# For each day, count encounters where adm <= day < dc
def occupancy_on(d):
    mask = (ward_enc["admission_date"] <= d) & (d < ward_enc["discharge_date"])
    return mask.sum() / WARD_CAPACITY[ward]

daily = pd.Series([occupancy_on(d) for d in dates], index=dates)

fig, ax = plt.subplots(figsize=(10, 4))
daily.plot(ax=ax, color="#EE7F00")
ax.set_title(f"Daily bed occupancy · {ward} · Q1 2026")
ax.set_ylabel("Occupancy")
ax.set_xlabel("Date")
ax.axhline(1.0, color="grey", linestyle="--", linewidth=1, label="Capacity")
ax.set_ylim(0, 1.4)
ax.legend()
plt.tight_layout()
plt.show()

> **Try it.** Reproduce the same chart for `"ICU"` rather than `"Cardiology"`. Are there days when the ICU was over capacity (above the dashed line)? *(Change one line: `ward = "ICU"`.)*

In [ ]:
# Histogram: distribution of length of stay
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(enc["los_days"], bins=range(0, 25), color="#2e8b57", edgecolor="white")
ax.set_title("Length of stay distribution · Saint Mary's Q1 2026")
ax.set_xlabel("Length of stay (days)")
ax.set_ylabel("Number of cases")
ax.axvline(enc["los_days"].mean(), color="red", linestyle="--",
           label=f"mean = {enc['los_days'].mean():.1f} days")
ax.legend()
plt.tight_layout()
plt.show()

> **Try it.** Add a second vertical line for the *median* LOS in a different colour, with `linestyle=':'`. Compare the position of the mean and median lines — what does the gap between them tell you about the shape of the distribution?

### Three plot types you need

| Type | Question it answers | Pandas call |
|---|---|---|
| **bar / barh** | "Ranking" — which is biggest, smallest? | `series.plot(kind="bar")` |
| **line** | "Trend" — how does it change over time? | `series.plot()` (default for time-indexed) |
| **hist** | "Distribution" — what does the spread look like? | `ax.hist(series)` |

For a foundations course, this is enough. Pie charts are almost never the right answer.

## ⏸️ Mini-Challenge 2 — Visualise (25 min)

Open `challenges/ch2_visualise.ipynb`. Build the chart the CMO will see in 5 seconds. *Fast-Track:* add a break-even reference line and color-code positive vs. negative bars.

## §5 — Klinik: Saint Mary's Q1 Dashboard

End-to-end. Three numbers, three pictures. The notebook the CMO will read on Tuesday.

In [ ]:
# Number 1 — Bed occupancy by ward (Q1 average)
WARD_CAPACITY = {"ICU": 16, "InternalMedicine": 60, "Surgery": 40,
                 "Cardiology": 30, "Geriatrics": 28, "Pulmonology": 20, "Neurology": 18}

def avg_occupancy(ward_name):
    days = pd.date_range("2026-01-01", "2026-03-31", freq="D")
    ward_enc = enc[enc["ward"] == ward_name]
    rates = []
    for d in days:
        mask = (ward_enc["admission_date"] <= d) & (d < ward_enc["discharge_date"])
        rates.append(mask.sum() / WARD_CAPACITY[ward_name])
    return np.mean(rates)

occ_summary = pd.Series({w: avg_occupancy(w) for w in WARD_CAPACITY}).sort_values(ascending=False)
print(occ_summary.round(3))

> **Try it.** Read the occupancy rate of `"Geriatrics"` with `occ_summary.loc["Geriatrics"]`. Is it above the 85% target line we will draw shortly?

In [ ]:
# Number 2 — Total margin by DRG (top 5 contributors and top 5 drains)
margin_by_drg = (enc.groupby(["drg_code","label"])["margin_eur"]
                    .sum()
                    .sort_values()
                    .round(2))
print("--- Top 5 drains ---"); print(margin_by_drg.head(5))
print("--- Top 5 contributors ---"); print(margin_by_drg.tail(5))

> **Try it.** Compute the total margin across all DRGs as a single number with `margin_by_drg.sum()`. Is the hospital net profitable for Q1?

In [ ]:
# Number 3 — Mean ER wait time by triage level (this dataset is Q1 too)
er = pd.read_csv(DATA / "er_waittimes.csv", parse_dates=["timestamp"])
wait_by_triage = er.groupby("triage_level")["wait_minutes"].agg(["mean","median","count"]).round(1)
wait_by_triage

> **Try it.** What share of triage-1 cases (the highest priority) waited longer than 30 minutes? *(Build a mask `(er["triage_level"] == 1) & (er["wait_minutes"] > 30)`, take `.sum()`, divide by the number of triage-1 cases.)*

In [ ]:
# Picture 1 — Occupancy bar chart
fig, ax = plt.subplots(figsize=(9, 5))
occ_summary.plot(kind="barh", ax=ax, color="#EE7F00")
ax.set_title("Average bed occupancy by ward · Q1 2026")
ax.set_xlabel("Occupancy rate")
ax.axvline(0.85, color="grey", linestyle="--", label="Target 85%")
ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Picture 2 — Margin contributors / drains
top_5_loss = margin_by_drg.head(5)
top_5_gain = margin_by_drg.tail(5)
combined = pd.concat([top_5_loss, top_5_gain]).sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#c0392b" if v < 0 else "#2e8b57" for v in combined.values]
combined.plot(kind="barh", ax=ax, color=colors)
ax.set_title("Margin contribution by DRG · Q1 2026 (top 5 each side)")
ax.set_xlabel("Total margin (EUR)")
ax.axvline(0, color="black", linewidth=1)
plt.tight_layout(); plt.show()

In [ ]:
# Picture 3 — ER wait time distribution by triage
fig, ax = plt.subplots(figsize=(10, 4))
for tri, group in er.groupby("triage_level"):
    ax.hist(group["wait_minutes"], bins=range(0, 300, 15),
            alpha=0.5, label=f"Triage {tri}")
ax.set_title("ER wait time distribution by triage level · Q1 2026")
ax.set_xlabel("Wait time (minutes)"); ax.set_ylabel("Number of cases")
ax.legend()
plt.tight_layout(); plt.show()

> **Try it.** Save this ER wait-time figure to disk: insert `plt.savefig("er_waits_q1.png", dpi=200, bbox_inches="tight")` *before* `plt.show()`. Open the resulting PNG — is it slide-ready, or do labels get cut off?

**The CMO has her dashboard.** Three numbers, three pictures, every line of code shows where it came from. That's what 'analytics' means in practice.

## §6 — Clean code for notebooks

Your exam notebook on Tuesday will be read by someone (the examiner) who is **not you**, and **not in your head**. Make their life easy.

### A clean notebook has:

1. **A title and one-line motivation** at the top, in markdown.
2. **A "setup" cell** with all imports + data loading. Run-once.
3. **Helper functions** defined together, near the top.
4. **Sections in markdown** that match the analysis flow.
5. **One conclusion per cell.** If a code cell does five things, split it.
6. **A final reflection cell** in markdown: what you learned, what you'd do next.

### A clean notebook does NOT have:

- Cells run out of order (rerun top-to-bottom before submitting!)
- Variables named `df`, `df2`, `tmp`, `xx`
- Commented-out code that "might come in handy"
- Print statements left from debugging
- Plots without titles or axis labels


> **Try it.** Open your most recent homework notebook from this week. Tick which of the six "has" items it already satisfies, and which "does NOT have" items you will need to clean up before submitting your project.

## ⏸️ Mini-Challenge 3 — Clean a notebook (15 min)

Open `challenges/ch3_clean_notebook.ipynb`. Refactor a working but messy notebook into something exam-ready.

## §7 — Exam requirements (12.05.2026, 09:00–13:30)

**Format:** Jupyter Notebook + brief oral defence.

**You must include:**
1. A clear research question or business question, in **one** markdown cell at the top.
2. Data loaded and cleaned (1-3 cells). Show that you handled missing values explicitly.
3. **At least one aggregation** (groupby or pivot).
4. **At least one chart** with title, labels, units.
5. Code organised into **functions** where it makes sense — not pure script.
6. A final **reflection** cell: what did you learn, what would you do next?

**Rubric:**

| Criterion | Weight |
|---|---|
| Clarity of question | 20% |
| Data preparation | 20% |
| Correctness of analysis | 30% |
| Readability & structure | 20% |
| Reflection | 10% |

**The dataset is your choice.** Public health data, your own activity log, anything tabular with at least 50 rows.

**Submit by Monday 11.05. 23:59** as `LASTNAME_FIRSTNAME_python_exam.ipynb` via eCampus.


## §8 — Differential diagnosis: visualization bugs

| Symptom | Cause | Fix |
|---|---|---|
| Plot is empty | NaN in `y` data | `df.dropna(subset=["y_col"])` first |
| Bars overlap or labels cut off | figure too small | larger `figsize=(10,6)` + `plt.tight_layout()` |
| Plot doesn't show | missing `plt.show()` or wrong context | always end with `plt.show()` |
| Axes are wrong scale | matplotlib auto-scaled to outliers | `ax.set_ylim(0, 100)` explicitly |
| Legend missing | forgot `label="..."` in plot call | add `label=` and `ax.legend()` |

---

## Take-home for the exam

Your exam notebook is essentially the homework you've been doing all week, polished. Pick:
- **A dataset you care about** (your gym log, household expenses, exam scores, public health data)
- **One question** you actually want answered
- **One picture** that tells the answer

Keep it small and clean. **A 15-cell notebook beats a 50-cell mess every time.**

## Cliffhanger → exam (Tuesday)

You can now load, clean, transform, group, join, and visualise tabular data. You've practised this on Saint Mary's Hospital for three days. **You are ready.**

The exam is not a trap — it's a chance to apply this on data you actually care about.

See you on Tuesday. Bring coffee.